In [0]:
data = {
'sexo': ['Hombre', 'Mujer', 'No_Declarado', 'Hombre', 'Hombre', 'Mujer', 
             'No_Declarado', 'Mujer', 'Hombre', 'Mujer', 'Hombre'],
'nivel_educativo': ['Bachillerato', 'Licenciatura', 'Maestría', 'Licenciatura', 'Bachillerato',
'Maestría', 'Licenciatura', 'Bachillerato', 'Bachillerato', 'Licenciatura', 'Licenciatura'],
'edad': [25, 45, 65, 35, 22,67,20,31,45,36,70],
        'salario': [20000, 50000, 100000, 35000, 27000,40000,15000,18000,13000,40000,21000]
}

data=list(zip(data["sexo"],data["nivel_educativo"],data["edad"],data["salario"]))
df_train = spark.createDataFrame(data,["sexo","nivel_educativo","edad","salario"])
df_train.display()

In [0]:
data_test = {
'sexo': ['Hombre', 'Mujer', 'No_Declarado','Mujer',"No sé"],
'nivel_educativo': ['Secundaria', 'Licenciatura', 'Maestría','Bachillerato'],
'edad': [18,29,75,39],
        'salario': [8000,15000,17000,25000]
}

data_test=list(zip(data_test["sexo"],data_test["nivel_educativo"],data_test["edad"],data_test["salario"]))
df_test = spark.createDataFrame(data_test,["sexo","nivel_educativo","edad","salario"])
df_test.display()

In [0]:
from pyspark.ml.feature import OneHotEncoder,StringIndexer,VectorAssembler,CountVectorizer
from pyspark.ml import Pipeline

In [0]:
string_idx=StringIndexer(inputCols=["sexo","nivel_educativo"],outputCols=["idx_sexo","idx_nivel_educativo"],handleInvalid="keep",stringOrderType="alphabetAsc")
ohe=OneHotEncoder(inputCol="idx_sexo",outputCol="ohe_sexo",dropLast=False)
pipe=Pipeline(stages=[string_idx,ohe])
pipe.fit(df_train).transform(df_train).display()
[1,0,0,0]

In [0]:
string_idx=StringIndexer(inputCols=["sexo","nivel_educativo"],outputCols=["idx_sexo","idx_nivel_educativo"],handleInvalid="skip",stringOrderType="alphabetAsc")
"""
skip,error
"frequencyDesc"categorías mas frecuentes tienen índices más bajos. 
"frequencyAsc"  categorías menos frecuentes tienen índices más bajos.
"alphabetDesc"  orden alfabético inverso.
"alphabetAsc"  orden alfabético .
"""
ohe=OneHotEncoder(inputCols=["idx_sexo"],outputCols=["ohe_sexo"],dropLast=False)
assembler = VectorAssembler(
    inputCols=["idx_nivel_educativo","ohe_sexo"], 
    outputCol="features"
)
pip=Pipeline(stages=[string_idx,ohe,assembler])
df_ohe=pip.fit(df_train).transform(df_test)
df_ohe.display()

In [0]:
from pyspark.ml.feature import StandardScaler,MinMaxScaler

In [0]:
vec_assembler = VectorAssembler(inputCols=["salario"], outputCol="salario_vec")
standar_sca=StandardScaler(inputCol="salario_vec",outputCol="std_salario",withMean=True, withStd=True)
norm_salario=MinMaxScaler(inputCol="salario_vec",outputCol="norm_salario")
pipe=Pipeline(stages=[vec_assembler,standar_sca,norm_salario])
pipe_salario=pipe.fit(df_train).transform(df_test)
pipe_salario.display()